# EWSF demo: OOB weights, drift detection, and soft probabilities

This notebook demonstrates training the EWSF model, computing OOB-based weights, inducing drift, running `adapt_weights!`, and plotting p-values and weights.

In [ ]:
using Pkg
Pkg.activate("..")    # adjust depending on where you put the notebook (assumes scripts/ inside package root)
Pkg.instantiate()
using EWSF, CSV, DataFrames, Random, Plots, Statistics
Plots.backend() # display backend info


In [ ]:
# Generate synthetic training data
function generate_data(n=1000)
    Random.seed!(123)
    age = randn(n) .* 10 .+ 50
    chol = randn(n) .* 30 .+ 200
    smoker = rand(n) .< 0.3
    smoker_s = ifelse.(smoker, "yes", "no")
    target = [ (age[i] > 52 || chol[i] > 240 || smoker[i]) ? "disease" : "healthy" for i in 1:n ]
    df = DataFrame(age=age, cholesterol=chol, smoker=smoker_s, target=target)
    return df
end

train = generate_data(800)
test = generate_data(200)

CSV.write("train_synthetic.csv", train)
CSV.write("test_synthetic.csv", test)
println("Data generated and saved to CSVs")


In [ ]:
# Preprocess and train
train_clean, enc = EWSF.EWSFData.data_preprocess(train, "target", ["age","cholesterol","smoker"])
model = EWSF.EWSFModel.train_ewsf(train_clean, enc; n_trees=60, n_subforests=3, max_depth=6)
println("Trained. Subforest weights (OOB-initialized): ", [sf.weight for sf in model.subforests])


In [ ]:
# Create drifted dataset
function create_drifted(df)
    df2 = deepcopy(df)
    df2.age .= df2.age .+ 5.0   # mean shift in age
    for i in 1:nrow(df2)
        if rand() < 0.2
            df2.smoker[i] = "yes"
        end
    end
    return df2
end

incoming = create_drifted(test)
println("Drift induced in incoming data (age shifted + smoker rate increased)")


In [ ]:
# Build preprocessed forms for drift adaptation
train_pre = train_clean
# Map incoming to encoded numeric values similar to preprocessing
map_smoker = enc.feature_encoders[:smoker]
new_df = DataFrame(age = Float64[], cholesterol=Float64[], smoker=Int[])
for r in eachrow(incoming)
    push!(new_df, (Float64(r.age), Float64(r.cholesterol), get(map_smoker, r.smoker, 0)))
end
rename!(new_df, [:age, :cholesterol, :smoker])

summary_before = EWSF.EWSFModel.feature_drift_pvalues(model, train_pre, new_df; n_perm=200)
pvals_before = summary_before[1]
println("Feature p-values (before adapt):")
for (k,v) in pvals_before
    println(k, " => ", round(v, digits=4))
end


In [ ]:
# Run adapt_weights! which changes subforest weights based on drift
summary = EWSF.EWSFModel.adapt_weights!(model, train_pre, new_df; p_threshold=0.05, n_perm=200, lambda=3.0)
println("Adaptation summary:\n", summary)
println("New subforest weights: ", summary["new_weights"])

In [ ]:
# Plot p-values before and report new weights
feature_names = collect(keys(pvals_before))
vals = [ pvals_before[f] for f in feature_names ]
bar(string.(feature_names), vals, title="Permutation p-values (train vs incoming)", ylabel="p-value", xlabel="feature")
hline!([0.05], linestyle=:dash, label="p=0.05")


In [ ]:
# Evaluate model before & after adaptation on incoming data (soft predictions)
pred_probs = EWSF.EWSFModel.predict_model(model, incoming)
first(pred_probs, 5)


In [ ]:
# Compute accuracy using predicted label vs ground truth on incoming
function compute_accuracy(pred_df, true_df)
    preds = pred_df[!, :prediction]
    true = true_df[!, :target]
    return sum(string(preds[i]) == string(true[i]) for i in 1:length(preds)) / length(preds)
end
acc_incoming = compute_accuracy(pred_probs, incoming)
println("Accuracy on incoming (post-adapt weights already applied): ", round(acc_incoming, digits=4))
